In [0]:
%run /Users/sandysakthivel2005@gmail.com/common/03_Logger


In [0]:
logger.info("Logger Working")
print(logger)

<Logger SocialMediaPipeline (WARNING)>


Logger notebook executed successfully


In [0]:
# MAGIC %run /Users/sandysakthivel2005@gmail.com/common/03_Logger

from pyspark.sql.functions import *

try:

    logger.info("Silver Tweets Pipeline Started")
    print("Silver Tweets Pipeline Started")

    # ==========================================
    # Read Bronze Streaming Table
    # ==========================================

    bronzeDF = (
        spark.readStream
             .table("bronze_catalog1.raw.bronze_tweets_raw")
    )

    logger.info("Bronze Tweets Table Read Successfully")
    print("Bronze Tweets Table Read Successfully")

    # ==========================================
    # Remove Duplicates
    # ==========================================

    silverDF = bronzeDF.dropDuplicates(["tweet_id"])

    # ==========================================
    # Handle Null Values
    # ==========================================

    silverDF = (
        silverDF.fillna({
            "likes": 0,
            "retweets": 0,
            "replies": 0,
            "impressions": 0,
            "engagement": 0
        })
    )

    # ==========================================
    # Data Validation
    # ==========================================

    silverDF = (
        silverDF
            .filter(col("tweet_id").isNotNull())
            .filter(col("user_id").isNotNull())
            .filter(col("tweet_text").isNotNull())
            .filter(col("timestamp_1").isNotNull())
            .filter(col("likes") >= 0)
            .filter(col("retweets") >= 0)
            .filter(col("replies") >= 0)
            .filter(col("impressions") >= 0)
            .filter(col("engagement") >= 0)
    )

    # ==========================================
    # Standardize Text
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("tweet_text", trim(col("tweet_text")))
    )

    # ==========================================
    # Convert Data Types
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("user_id", col("user_id").cast("long"))
            .withColumn("likes", col("likes").cast("int"))
            .withColumn("retweets", col("retweets").cast("int"))
            .withColumn("replies", col("replies").cast("int"))
            .withColumn("impressions", col("impressions").cast("int"))
            .withColumn("engagement", col("engagement").cast("double"))
    )

    # ==========================================
    # Standardize Date
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("tweet_date", to_date(col("timestamp_1")))
    )

    # ==========================================
    # Audit Column
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("silver_load_time", current_timestamp())
            .withColumn("pipeline_name", lit("Silver_Tweets"))
    )

    logger.info("Silver Tweets Transformations Completed Successfully")
    print("Silver Tweets Transformations Completed Successfully")

    # ==========================================
    # Write Silver Table
    # ==========================================

    silverQuery = (
        silverDF.writeStream
            .trigger(availableNow=True)
            .format("delta")
            .outputMode("append")
            .option(
                "checkpointLocation",
                "abfss://socialmedia@socialmediaadls001.dfs.core.windows.net/checkpoints/silver_tweets"
            )
            .option("mergeSchema", "true")
            .toTable("silver_catalog1.processed.silver_tweets")
    )

    silverQuery.awaitTermination()

    logger.info("Silver Tweets Loaded Successfully")
    print("Silver Tweets Loaded Successfully")

except Exception as e:

    logger.error(f"Silver Tweets Pipeline Failed: {str(e)}")
    print(f"Silver Tweets Pipeline Failed: {str(e)}")
    raise

Silver Tweets Pipeline Started
Bronze Tweets Table Read Successfully
Silver Tweets Transformations Completed Successfully


In [0]:
%sql
SELECT COUNT(*)
FROM silver_catalog1.processed.silver_tweets;

count(1)
2612


In [0]:
%sql
SELECT *
FROM silver_catalog1.processed.silver_tweets
LIMIT 20;

tweet_id,user_id,tweet_text,timestamp,timestamp_1,likes,retweets,replies,impressions,engagement,bronze_load_time,pipeline_name,source_system,ingestion_date,tweet_date,silver_load_time
T26417,6353,Worst service ever :(,api,2025-01-15T17:13:00Z,0,283,7,813,1059.0,2026-07-09T09:52:24.541Z,Silver_Tweets,Azure Event Hub,2026-07-09,2025-01-15,2026-07-10T04:05:00.227Z
T39272,3665,Worst service ever :(,android,2025-01-17T07:30:00Z,13696,247,935,1082,411.0,2026-07-09T09:52:24.541Z,Silver_Tweets,Azure Event Hub,2026-07-09,2025-01-17,2026-07-10T04:05:00.227Z
T9431,1589,Spark > Hadoop ???,web,2025-01-18T22:36:00Z,12217,2351,908,2435,759.0,2026-07-09T09:52:24.541Z,Silver_Tweets,Azure Event Hub,2026-07-09,2025-01-18,2026-07-10T04:05:00.227Z
T15738,7651,Data pipeline broke @ midnight!!!,ios,2025-01-06T00:13:00Z,5930,4802,443,1168,1009.0,2026-07-09T09:52:24.541Z,Silver_Tweets,Azure Event Hub,2026-07-09,2025-01-06,2026-07-10T04:05:00.227Z
T9919,3644,Data pipeline broke @ midnight!!!,android,2025-01-08T02:47:00Z,1320,3659,340,258,979.0,2026-07-09T09:52:24.541Z,Silver_Tweets,Azure Event Hub,2026-07-09,2025-01-08,2026-07-10T04:05:00.227Z
T5929,9713,Spark > Hadoop ???,web,2025-01-22T09:35:00Z,5897,3077,275,505,752.0,2026-07-09T09:52:24.541Z,Silver_Tweets,Azure Event Hub,2026-07-09,2025-01-22,2026-07-10T04:05:00.227Z
T26041,3425,"""NaN""",ios,2025-01-16T06:42:00Z,15237,323,721,1578,1197.0,2026-07-09T09:52:24.541Z,Silver_Tweets,Azure Event Hub,2026-07-09,2025-01-16,2026-07-10T04:05:00.227Z
T24418,2238,Error###Detected,android,2025-01-21T05:28:00Z,12681,3609,901,205,807.0,2026-07-09T09:52:24.541Z,Silver_Tweets,Azure Event Hub,2026-07-09,2025-01-21,2026-07-10T04:05:00.227Z
T9521,2059,Spark > Hadoop ???,android,2025-01-07T03:06:00Z,12507,1245,74,496,1659.0,2026-07-09T09:52:24.541Z,Silver_Tweets,Azure Event Hub,2026-07-09,2025-01-07,2026-07-10T04:05:00.227Z
T22094,4105,Null?? value??,ios,2025-01-14T12:17:00Z,16925,4970,859,2793,430.0,2026-07-09T09:52:24.541Z,Silver_Tweets,Azure Event Hub,2026-07-09,2025-01-14,2026-07-10T04:05:00.227Z


In [0]:
%sql
SELECT COUNT(*)
FROM silver_catalog1.processed.silver_tweets
WHERE tweet_id IS NULL;

count(1)
0


In [0]:
%sql
SELECT COUNT(*) FROM silver_catalog1.processed.silver_tweets;

count(1)
2612
